<a href="https://colab.research.google.com/github/akshat-rakheja/RAG-Chatbot-/blob/main/RAG_Chatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# pip install libraries
!pip install sentence-transformers faiss-cpu langchain
!pip install pypdf langchain_community
!pip install langchain-text-splitters

In [ ]:
# upload the document
from google.colab import files
uploaded = files.upload()
for filename in uploaded.keys():
  print("uploaded:",filename)

In [ ]:
# document ingestion pipeline
from langchain_community.document_loaders import PyPDFLoader
pdf_file = list(uploaded.keys())[0]
loader = PyPDFLoader(pdf_file) # loader object
documents = loader.load()
print("Number of pages loaded",len(documents))


In [ ]:
print(documents[0])

In [ ]:
# Chunking
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500,chunk_overlap=100)
chunks = text_splitter.split_documents(documents)
print("total chunks created",len(chunks))

In [ ]:
# Displaying 1 chunk
print(chunks[0].page_content)

In [ ]:
# embeddings from the chunk
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

#defining the embedding model
model = SentenceTransformer('all-MiniLM-L6-V2') # 22M parameters only , smaller model

#creating the embeddings
embeddings = model.encode([chunk.page_content for chunk in chunks]) #for each chunks in the chunk

dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings))

print("Total vectors in the index ",index.ntotal)

In [ ]:
print(dimension)

In [ ]:
import matplotlib.pyplot as plt
def visualize_embedding(vector,chunk_text=None,max_text_chars=500):
  vector = np.array(vector)
  normalized = (vector - vector.min())/(vector.max()-vector.min())
  color_band = normalized.reshape(1,-1)

  if chunk_text is not None:
    print("Chunk text \n")
    print(chunk_text[:max_text_chars])
    print()

    plt.figure(figsize =(12,2))
    plt.imshow(color_band,aspect="auto",cmap="viridis")
    plt.yticks([])
    plt.xlabel(f"Embedding Dimensions ({len(vector)})")
    plt.title("vector representation")
    plt.show()

In [ ]:
visualize_embedding(embeddings[1],chunks[1].page_content)

In [ ]:
# Retrievel

query = input("Enter your question : ")
query_embedding = model.encode([query])
visualize_embedding(query_embedding,query)

In [ ]:
D,I = index.search(np.array(query_embedding),k=5) #looking for 5 similar items
retrieved_chunks = [chunks[i] for i in I[0]]
print("Top retrieved chunks : \n")

for chunk in retrieved_chunks:
  print(chunk.page_content)
  print("Page:",chunk.metadata['page'])
  print("------------------------------------------------")


In [ ]:
# Performing Reranking
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
pairs = [(query,chunk.page_content) for chunk in retrieved_chunks]
scores = reranker.predict(pairs)

ranked_chunks = sorted(zip(scores,retrieved_chunks),
                       key = lambda x:x[0],
                       reverse = True)
print("After reranking :\n")
for score,chunk in ranked_chunks :
  print("score :",score)
  print(chunk.page_content)
  print("Page number :",chunk.metadata["page"])
  print("-------------------------------------------")


In [ ]:
# Combine rerank chunks with context
top_chunks = [chunk.page_content for _,chunk in ranked_chunks[:3]]
context = "\n\n".join(top_chunks)
print("Context : \n")
print(context)

In [ ]:
import google.generativeai as genai
from google.colab import userdata

# 1. Configure Gemini API
GEMINI_API_KEY = userdata.get('GeminiAPI_Key')
genai.configure(api_key=GEMINI_API_KEY)

# 2. Initialize the model with the correct preview string
# 'gemini-3-flash-preview' is the current standard for the free tier
model = genai.GenerativeModel('gemini-3-flash-preview')

# 3. Define your RAG logic
def generate_rag_answer(context, query):
    prompt = f"""
    You are a precise assistant. Answer the question using ONLY the provided context.
    If the answer is not contained within the context, state: "Information not found in document."

    Context:
    {context}

    Question:
    {query}

    Answer:
    """

    try:
        response = model.generate_content(prompt)
        return response.text
    except Exception as e:
        return f"An error occurred: {e}"

result = generate_rag_answer(context, query)
print(result)

In [ ]:
def evaluate_response(query, context, response):
    eval_prompt = f"""
    Evaluate the following RAG response based on the Context.
    Score from 0 to 1 (0 is fail, 1 is perfect).

    1. Faithfulness: Is the answer derived ONLY from the context?
    2. Relevancy: Does the answer actually address the query?

    Query: {query}
    Context: {context}
    Response: {response}

    Return ONLY a JSON: {{"faithfulness": score, "relevancy": score}}
    """
    eval_result = model.generate_content(eval_prompt)
    return eval_result.text



In [ ]:
evaluation_result = evaluate_response(query, context, result)
print(evaluation_result)
